# Causal Matrix Inspection
Visualise one rollout's sentence-sentence causal influence matrix alongside receiver-head R-scores and Thought Anchor sentence labels.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path('.').resolve().parent

DATASET = 'openai_humaneval'
CAUSAL_DIR = PROJECT_ROOT / 'results' / 'causal_matrices_openai_humaneval_qwen3_5_0_8b'
RECEIVER_HEAD_FILE = PROJECT_ROOT / 'results' / 'receiver_head_summary_humaneval_qwen3_5_0_8b.jsonl'
LABELS_FILE = PROJECT_ROOT / 'results' / 'sentence_labels_humaneval_qwen3_5_0_8b.jsonl'

In [ ]:
# Pick the first available .npz
npz_files = sorted(CAUSAL_DIR.glob('*.npz'))
print(f'Found {len(npz_files)} causal matrices')
npz_path = npz_files[0]
data = np.load(npz_path, allow_pickle=True)
causal_matrix = data['causal_matrix']          # [M, M]
task_id = str(data['task_id'])
sample_id = int(data['sample_id'])
M = int(data['num_sentences'])
print(f'task_id={task_id!r}, sample_id={sample_id}, M={M}')

In [ ]:
# Load matching receiver-head summary
rh_scores = None
if RECEIVER_HEAD_FILE.exists():
    with RECEIVER_HEAD_FILE.open() as fh:
        for line in fh:
            row = json.loads(line)
            if str(row.get('task_id')) == task_id and int(row.get('sample_id', -1)) == sample_id:
                rh_scores = row.get('sentence_scores', [])
                break
    if rh_scores:
        print(f'Receiver-head sentence_scores: {len(rh_scores)} entries')
    else:
        print('No matching receiver-head row found')

In [ ]:
# Load matching sentence labels
labels = None
TAG_COLORS = {
    'problem_decomposition': '#e6194b',
    'example_analysis': '#f58231',
    'algorithm_design': '#ffe119',
    'code_writing': '#3cb44b',
    'verification': '#42d4f4',
    'error_correction': '#4363d8',
    'solution_summary': '#911eb4',
    'other': '#a9a9a9',
}
if LABELS_FILE.exists():
    with LABELS_FILE.open() as fh:
        for line in fh:
            row = json.loads(line)
            if str(row.get('task_id')) == task_id and int(row.get('sample_id', -1)) == sample_id:
                labels = row.get('labels', {})
                break
    print(f'Labels found: {labels is not None}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [3, 1]})

# ── Causal matrix heatmap ──────────────────────────────────────────────────────
ax = axes[0]
vmax = np.nanpercentile(np.abs(causal_matrix), 95)
im = ax.imshow(causal_matrix, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
ax.set_title(f'Causal Matrix — {task_id} s{sample_id}', fontsize=11)
ax.set_xlabel('Source sentence (suppressed)')
ax.set_ylabel('Target sentence')
plt.colorbar(im, ax=ax, shrink=0.8, label='mean log-KL (col-normalised)')

# Colour tick labels by dominant tag if labels are available
if labels:
    ticks = list(range(M))
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    tick_colors = []
    for idx in ticks:
        entry = labels.get(str(idx + 1), {})
        tags = entry.get('function_tags', ['other'])
        dominant = tags[0] if tags else 'other'
        tick_colors.append(TAG_COLORS.get(dominant, '#a9a9a9'))
    for label, color in zip(ax.get_xticklabels(), tick_colors):
        label.set_color(color)
    for label, color in zip(ax.get_yticklabels(), tick_colors):
        label.set_color(color)

# ── Receiver-head R-scores bar chart ──────────────────────────────────────────
ax2 = axes[1]
if rh_scores and len(rh_scores) == M:
    ax2.barh(range(M), rh_scores, color='steelblue', alpha=0.75)
    ax2.set_yticks(range(M))
    ax2.set_yticklabels(range(M))
    ax2.invert_yaxis()
    ax2.set_xlabel('R-score')
    ax2.set_title('Receiver-head scores')
else:
    ax2.text(0.5, 0.5, 'No receiver-head\ndata', ha='center', va='center', transform=ax2.transAxes)
    ax2.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation scatter: causal influence vs receiver-head score
if rh_scores and len(rh_scores) == M:
    col_means = np.nanmean(causal_matrix, axis=0)  # mean influence received per sentence j
    x = np.array(rh_scores)
    y = col_means
    mask = ~np.isnan(y)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(x[mask], y[mask], alpha=0.7)
    ax.set_xlabel('Receiver-head R-score (sentence)')
    ax.set_ylabel('Mean column KL (causal influence received)')
    ax.set_title('Causal influence vs R-score')
    corr = np.corrcoef(x[mask], y[mask])[0, 1]
    ax.text(0.05, 0.95, f'r = {corr:.3f}', transform=ax.transAxes, va='top')
    plt.tight_layout()
    plt.show()